In [ ]:
"""Feature Engineering     data["HouseAge"] = (
        data["YrSold"] - data["YearBuilt"]
    ) 추가해보니 성능향상됌


    추가로 화장실 정보를 합쳐보니 0.14745 까지 향상
    
    feature 한개 더 추가하니까 0.17297까지 다시 떨어짐
    """


'Feature Engineering     data["HouseAge"] = (\n        data["YrSold"] - data["YearBuilt"]\n    ) 추가해보니 성능향상됌\n\n\n    추가로 화장실 정보를 합쳐보니 0.14745 까지 향상\n    \n    '

In [8]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

#최초의 신경망 

class HousePriceMLP(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128,64)
        self.fc3 = nn.Linear(64, 1)

        self.relu = nn.ReLU()

    def forward(self,x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)        
        x = self.fc3(x)

        return x.squeeze(1)


print(train.shape)
print(test.shape)
print(train.head())

(1460, 81)
(1459, 80)
   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
1         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      5   
2         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      9   
3         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
4         Lvl    AllPub  ...        0    NaN   NaN         NaN       0     12   

  YrSold  SaleType  SaleCondition  SalePrice  

In [9]:
#현재: X와 y 분리
y = train["SalePrice"]
x = train.drop(columns=["Id","SalePrice"])

test_id = test["Id"]
x_test = test.drop(columns=["Id"])

print("x크기",x.shape)
print("y크기",y.shape)
print("test크기",x_test.shape)

x크기 (1460, 79)
y크기 (1460,)
test크기 (1459, 79)


In [10]:
def add_features(data):
    data = data.copy()

    data["HouseAge"] = (
        data["YrSold"] - data["YearBuilt"]
    )

    data["TotalBath"] = (
        data["FullBath"].fillna(0)
        + 0.5 * data["HalfBath"].fillna(0)
        + data["BsmtFullBath"].fillna(0)
        + 0.5 * data["BsmtHalfBath"].fillna(0)
    )

    data["QualityGrLivArea"] = (
        data["OverallQual"] * data["GrLivArea"]
    )

    return data 
x = add_features(x)
x_test = add_features(x_test)

In [11]:
#MLP에 넣으려면 최종적으로 x가 모든 값이 숫자& 결측값이 없어야함!
#문자형 43개, 정수형 33개, 실수형 3개
#결측값 해석 수영장 1460중에 1453개 없음
print(x.dtypes.value_counts())
print(x.isnull().sum().sort_values(ascending=False).head(20))

object     43
int64      35
float64     4
Name: count, dtype: int64
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageFinish      81
GarageYrBlt       81
GarageQual        81
GarageCond        81
GarageType        81
BsmtExposure      38
BsmtFinType2      38
BsmtQual          37
BsmtFinType1      37
BsmtCond          37
MasVnrArea         8
Electrical         1
WoodDeckSF         0
dtype: int64


In [12]:
#첫번째 베이스라인에서는 너무 복잡하게 하지말고
#범주형 결측값 -> Missing
#수치형 결측값 -> 중앙값

In [13]:
"""지금 바로 결측값을 채우면 안 되는 이유

먼저 train 데이터를 다시 학습용과 검증용으로 나눠야 해.

x
├── x_train: 모델이 공부할 문제
└── x_valid: 모델이 풀어볼 시험문제

y
├── y_train: 공부할 문제의 정답
└── y_valid: 시험문제의 정답

결측값을 먼저 전체 데이터의 중앙값으로 채우면 validation 
데이터의 정보가 training에 조금 들어갈 수 있어. 
이것을 데이터 누수라고 해.

집값은 너무 비싸기 때문에 log로 취해서 사용
"""



'지금 바로 결측값을 채우면 안 되는 이유\n\n먼저 train 데이터를 다시 학습용과 검증용으로 나눠야 해.\n\nx\n├── x_train: 모델이 공부할 문제\n└── x_valid: 모델이 풀어볼 시험문제\n\ny\n├── y_train: 공부할 문제의 정답\n└── y_valid: 시험문제의 정답\n\n결측값을 먼저 전체 데이터의 중앙값으로 채우면 validation \n데이터의 정보가 training에 조금 들어갈 수 있어. \n이것을 데이터 누수라고 해.\n\n집값은 너무 비싸기 때문에 log로 취해서 사용\n'

In [14]:
y_log = np.log1p(y)

from sklearn.model_selection import train_test_split

x_train, x_valid, y_train, y_valid = train_test_split( 
    x,
    y_log,
    test_size=0.2,
    random_state=42
)



In [15]:
numeric_columns = x_train.select_dtypes(
    include=["int64","float64"]
).columns

categorical_columns = x_train.select_dtypes(
    include=["object"]
).columns

print("수치형:", len(numeric_columns))
print("범주형:", len(categorical_columns))

수치형: 39
범주형: 43


In [16]:
numeric_medians = x_train[numeric_columns].median()

x_train[numeric_columns] = (
    x_train[numeric_columns].fillna(numeric_medians)
)

x_valid[numeric_columns] = (
    x_valid[numeric_columns].fillna(numeric_medians)
)

x_test[numeric_columns] = (
    x_test[numeric_columns].fillna(numeric_medians)
)

x_train[categorical_columns] = (
    x_train[categorical_columns].fillna("Missing")
)

x_valid[categorical_columns] = (
    x_valid[categorical_columns].fillna("Missing")
)

x_test[categorical_columns] = (
    x_test[categorical_columns].fillna("Missing")
)

In [17]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_columns),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_columns
        )
    ]
)

x_train_processed = preprocessor.fit_transform(x_train)
x_valid_processed = preprocessor.transform(x_valid)
x_test_processed = preprocessor.transform(x_test)

print("처리 후 train:", x_train_processed.shape)
print("처리 후 valid:", x_valid_processed.shape)
print("처리 후 test:", x_test_processed.shape)

print("train 결측값:", np.isnan(x_train_processed).sum())
print("valid 결측값:", np.isnan(x_valid_processed).sum())
print("test 결측값:", np.isnan(x_test_processed).sum())

처리 후 train: (1168, 304)
처리 후 valid: (292, 304)
처리 후 test: (1459, 304)
train 결측값: 0
valid 결측값: 0
test 결측값: 0


In [18]:
x_train_tensor = torch.tensor(
    x_train_processed,
    dtype=torch.float32
)

x_valid_tensor = torch.tensor(
    x_valid_processed,
    dtype=torch.float32
)

x_test_tensor = torch.tensor(
    x_test_processed,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train.values,
    dtype=torch.float32
)

y_valid_tensor = torch.tensor(
    y_valid.values,
    dtype=torch.float32
)

print(x_train_tensor.shape)
print(y_train_tensor.shape)
print(x_train_tensor.dtype)


torch.Size([1168, 304])
torch.Size([1168])
torch.float32


In [19]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(
    x_train_tensor,
    y_train_tensor
)

valid_dataset = TensorDataset(
    x_valid_tensor,
    y_valid_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=True
)

for batch_x, batch_y in train_loader:
    print("한 배치의 입력:", batch_x.shape)
    print("한 배치의 정답:", batch_y.shape)
    break

한 배치의 입력: torch.Size([32, 304])
한 배치의 정답: torch.Size([32])


In [20]:
input_dim = x_train_tensor.shape[1]
#주택 한 채당 특성 개수
print("입력 특성 개수:", input_dim)

입력 특성 개수: 304


In [21]:
model = HousePriceMLP(input_dim)

print(model)

HousePriceMLP(
  (fc1): Linear(in_features=304, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=1, bias=True)
  (relu): ReLU()
)


In [22]:
#criterion은 모델의 예측이 정답과 얼마나 다른지 계산하는 채점 기준
criterion = nn.MSELoss()

#optimizer 모델이 틀린 결과를 보고, 가중치를 어느 방향으로 얼마나 수정할지 실행하는 도구
optimizer = torch.optim.AdamW(
    model.parameters(), #모델 안의 가중치들을 수정 대상으로 지정
    lr=0.001, # 한번 학습할때 가중치를 얼마나 움직일지 결정
    weight_decay=0.0001 #가중치가 지나치게 커지는것을 막아 과적합을 줄임
)

In [23]:
"""
최고 점수 갱신
→ 기다린 횟수 0으로 초기화
→ 현재 가중치 저장
점수가 안좋아지면 patience_counter += 1
20번 연속 안좋아지면 학습종료
"""

best_valid_rmse = float("inf")
patience = 20
patience_counter = 0

In [24]:

for epoch in range(200):
    #model.train()이 중요해지는 것은 Dropout이나 BatchNorm -> 지금은 크게 상관 X
    model.train()

    total_train_loss = 0

    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()

        predictions = model(batch_x)

        loss = criterion(predictions, batch_y)
        
        loss.backward()

        optimizer.step()

        #loss.item(): Tensor인 loss를 일반 숫자로 꺼냄
        #batch_x.size(0): 현재 batch에 들어 있는 주택 수
        total_train_loss += loss.item() * batch_x.size(0)

    average_train_loss = total_train_loss / len(train_loader.dataset)

    model.eval()

    total_valid_loss = 0

    #validation에서는 모델을 평가만 하고 가중치를 수정X
    with torch.no_grad():
        for batch_x, batch_y in valid_loader:
            predictions = model(batch_x)

            loss = criterion(predictions, batch_y)

            total_valid_loss += loss.item() * batch_x.size(0)

    average_valid_loss = total_valid_loss / len(valid_loader.dataset)

    #평균 제곱근 오차, 낮을 수록 족음
    train_rmse = average_train_loss ** 0.5
    valid_rmse = average_valid_loss ** 0.5

    print(
        f"Epoch {epoch + 1}: "
        f"Train RMSE={train_rmse:.4f}, "
        f"Valid RMSE={valid_rmse:.4f}"
    )

    if valid_rmse < best_valid_rmse:
        best_valid_rmse = valid_rmse
        patience_counter = 0

        #파일에 저장
        #state_dict()는 모델이 현재 가지고 있는 학습된 가중치와 bias를 사전 형태로 모아주는 함수
        torch.save(
            model.state_dict(),
            "best_house_mlp.pt"
        )

        print(" -> Best model Saved!")
    
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print("Early Stopping!")
        break


model.load_state_dict(
    torch.load(
        "best_house_mlp.pt",
        #파일에서 모델 가중치처럼 안전한 데이터만 불러오겠다는 옵션
        weights_only=True
    )
)

model.eval()

print("Best Valid RMSE:", best_valid_rmse)

Epoch 1: Train RMSE=8.4711, Valid RMSE=3.1239
 -> Best model Saved!
Epoch 2: Train RMSE=1.3026, Valid RMSE=0.5031
 -> Best model Saved!
Epoch 3: Train RMSE=0.3131, Valid RMSE=0.2133
 -> Best model Saved!
Epoch 4: Train RMSE=0.2178, Valid RMSE=0.1895
 -> Best model Saved!
Epoch 5: Train RMSE=0.1889, Valid RMSE=0.1743
 -> Best model Saved!
Epoch 6: Train RMSE=0.1689, Valid RMSE=0.1688
 -> Best model Saved!
Epoch 7: Train RMSE=0.1551, Valid RMSE=0.1779
Epoch 8: Train RMSE=0.1488, Valid RMSE=0.1604
 -> Best model Saved!
Epoch 9: Train RMSE=0.1356, Valid RMSE=0.1632
Epoch 10: Train RMSE=0.1309, Valid RMSE=0.1567
 -> Best model Saved!
Epoch 11: Train RMSE=0.1217, Valid RMSE=0.1494
 -> Best model Saved!
Epoch 12: Train RMSE=0.1149, Valid RMSE=0.1443
 -> Best model Saved!
Epoch 13: Train RMSE=0.1165, Valid RMSE=0.1457
Epoch 14: Train RMSE=0.1092, Valid RMSE=0.1457
Epoch 15: Train RMSE=0.1083, Valid RMSE=0.1436
 -> Best model Saved!
Epoch 16: Train RMSE=0.1029, Valid RMSE=0.1404
 -> Best model 

In [25]:
model.eval()

with torch.no_grad():
    test_log_predictions = model(x_test_tensor)

test_predictions = np.expm1(
    test_log_predictions.cpu().numpy()
)

submission = pd.DataFrame({
    "Id": test_id,
    "SalePrice":test_predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)